In [ ]:
import os
import random
from pathlib import Path
import re
import pickle
import json
from datasets import load_dataset
import math

ALPHA = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
ALPHA_DOT = [alpha + '.' for alpha in ALPHA]
# parse option from gpt response
def parse_option(response):
    # the option is in between ** and **
    splits = response.split('ANSWER:')
    if len(splits) >= 2:
        option = splits[1].strip()
        for i in range(len(ALPHA_DOT)):
            if ALPHA_DOT[i] in option:
                option = ALPHA[i]
                return option
        for i in range(len(ALPHA)):
            if ALPHA[i] in option:
                option = ALPHA[i]
                return option
    else:
        return None
    
# current time
dataSet = 'mmmu' # choose from mmmu, clevr, textocr
dataSlice = 'val' # choose from val, dev
cacheSet = 'mmmu' # choose from mmmu, clevr, textocr
cacheSlice = 'dev' # choose from val, dev
model_name = '7B' # choose from 2B, 7B, 72Bint4
time = '20250129004208'
data_path = Path('../data/mmmu')
result_read_path = f'./results/Qwen_{model_name}_{dataSet}_{dataSlice}_{cacheSet}_{cacheSlice}_image_results_{time}.jsonl'
dataDir = '../data'

In [ ]:
if dataSet == 'mmmu':
    # load base dataset
    if dataSlice == 'val':
        test_dataset = load_dataset("lmms-lab/MMMU", split="validation")
    elif dataSlice == 'dev':
        test_dataset = load_dataset("lmms-lab/MMMU", split="dev")
    else:
        test_dataset = load_dataset("lmms-lab/MMMU", split="test")
    test_dataset_single_image = test_dataset.filter(lambda x: x['image_2'] is None)
    # sample 2000 examples from the test dataset
    test_dataset_single_image = test_dataset_single_image.shuffle(seed=42)
else:
    if dataSlice == 'val':
        data_file = os.path.join(dataDir, dataSet, 'support.json')
    else:
        data_file = os.path.join(dataDir, dataSet, 'query.json')
    with open(data_file, 'r') as f:
        query_meta = json.load(f)
    test_dataset_single_image = query_meta
    # sample 2000 examples from the test dataset
    random.shuffle(test_dataset_single_image)
    
# open pickle
with open(result_read_path, 'rb') as f:
    result_objs = pickle.load(f)

In [ ]:
# evaluate results
llm_first_choice_list = []
llm_second_choice_list = []
if dataSet == 'mmmu':
	for each in result_objs:
		llm_first_choice_list.append(parse_option(each['first_vlm_response'][0]))
		llm_second_choice_list.append(parse_option(each['second_vlm_response'][0]))
elif dataSet == 'clevr':
	for each in result_objs:
		vlm_ans = each['first_vlm_response'][0].lower().split('answer: ')[-1].strip().strip('."')
		vlm_ans = re.findall(r'\d+', vlm_ans)
		if vlm_ans:      
			llm_first_choice_list.append(vlm_ans[0])
		else:
			llm_first_choice_list.append(None)
		vlm_ans = each['second_vlm_response'][0].lower().split('answer: ')[-1].strip().strip('."')
		vlm_ans = re.findall(r'\d+', vlm_ans)
		if vlm_ans:      
			llm_second_choice_list.append(vlm_ans[0])
		else:
			llm_second_choice_list.append(None)
elif dataSet == 'textocr':
	for each in result_objs:
		ans_text = each['first_vlm_response'][0].lower().replace('<|im_end|>','')
		ans_text = ans_text.split('answer:')
		if len(ans_text) > 1:
			ans_text = ans_text[1]
		else:
			ans_text = ans_text[0]
		ans_text = ans_text.split(' is:')
		if len(ans_text) > 1:
			text = ans_text[1]
		else:
			ans_text = ans_text[0].split(' is')
			if len(ans_text) > 1:
				text = ans_text[1]
			else:
				text = ans_text[0]
		text = text.replace(' ','').replace('"','').replace('.','')
		llm_first_choice_list.append(text)

		ans_text = each['second_vlm_response'][0].lower().replace('<|im_end|>','')
		ans_text = ans_text.split('answer:')
		if len(ans_text) > 1:
			ans_text = ans_text[1]
		else:
			ans_text = ans_text[0]
		ans_text = ans_text.split(' is:')
		if len(ans_text) > 1:
			text = ans_text[1]
		else:
			ans_text = ans_text[0].split(' is')
			if len(ans_text) > 1:
				text = ans_text[1]
			else:
				text = ans_text[0]
		text = text.replace(' ','').replace('"','').replace('.','')
		llm_second_choice_list.append(text)


In [ ]:
# compare model response with the ground truth
vllm_count = 0
for each in result_objs:
    if 'icl_example' in each:
        vllm_count += 1
slice_size = math.floor(vllm_count/3)
vllm_count = 0
for idx,each in enumerate(result_objs):
    if vllm_count == slice_size:
        first_idx = idx
    elif vllm_count == slice_size * 2:
        second_idx = idx
    if 'icl_example' in each:
        vllm_count += 1

#first one-third:
one_third = test_dataset_single_image['answer'][:first_idx]
one_third_results = result_objs[:first_idx]
one_third_first = llm_first_choice_list[:first_idx]
one_third_second = llm_second_choice_list[:first_idx]
first_vllm_correct_count = 0
second_vllm_correct_count = 0
for idx, each in enumerate(one_third):
    if 'icl_example' in one_third_results[idx]:
        if str(one_third_first[idx]).lower() == str(each).lower():
            first_vllm_correct_count += 1
        if str(one_third_second[idx]).lower() == str(each).lower():
            second_vllm_correct_count += 1
# accuracy
print(first_vllm_correct_count/slice_size)
print(second_vllm_correct_count/slice_size)

#second one-third:
one_third = test_dataset_single_image['answer'][first_idx:second_idx]
one_third_results = result_objs[first_idx:second_idx]
one_third_first = llm_first_choice_list[first_idx:second_idx]
one_third_second = llm_second_choice_list[first_idx:second_idx]
first_vllm_correct_count = 0
second_vllm_correct_count = 0
for idx, each in enumerate(one_third):
    if 'icl_example' in one_third_results[idx]:
        if str(one_third_first[idx]).lower() == str(each).lower():
            first_vllm_correct_count += 1
        if str(one_third_second[idx]).lower() == str(each).lower():
            second_vllm_correct_count += 1
# accuracy
print(first_vllm_correct_count/slice_size)
print(second_vllm_correct_count/slice_size)

#last one-third:
one_third = test_dataset_single_image['answer'][second_idx:]
one_third_results = result_objs[second_idx:]
one_third_first = llm_first_choice_list[second_idx:]
one_third_second = llm_second_choice_list[second_idx:]
first_vllm_correct_count = 0
second_vllm_correct_count = 0
for idx, each in enumerate(one_third):
    if 'icl_example' in one_third_results[idx]:
        if str(one_third_first[idx]).lower() == str(each).lower():
            first_vllm_correct_count += 1
        if str(one_third_second[idx]).lower() == str(each).lower():
            second_vllm_correct_count += 1
# accuracy
print(first_vllm_correct_count/(vllm_count-2*slice_size))
print(second_vllm_correct_count/(vllm_count-2*slice_size))

In [ ]:
# compare model response with the ground truth
first_correct_count = 0
second_correct_count = 0
vllm_count = 0
first_vllm_correct_count = 0
second_vllm_correct_count = 0
for idx, each in enumerate(test_dataset_single_image):
    if not result_objs[idx]['id'] == each['id']:
        print(idx)
    if 'icl_example' in result_objs[idx]:
        vllm_count += 1
        if str(llm_first_choice_list[idx]).lower() == str(each['answer']).lower():
            first_correct_count += 1
            first_vllm_correct_count += 1
        if str(llm_second_choice_list[idx]).lower() == str(each['answer']).lower():
            second_correct_count += 1
            second_vllm_correct_count += 1
    else:
        if str(llm_first_choice_list[idx]).lower() == str(each['answer']).lower():
            first_correct_count += 1
        if str(llm_second_choice_list[idx]).lower() == str(each['answer']).lower():
            second_correct_count += 1

# accuracy
print(first_correct_count / len(test_dataset_single_image))
print(second_correct_count / len(test_dataset_single_image))
print(first_vllm_correct_count/vllm_count)
print(second_vllm_correct_count/vllm_count)